In [1]:
import os
import copy
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torchvision

from models import resnet20_cifar

ckpt_path = 'result/resnet20/model_best.pth.tar'
assert os.path.exists(ckpt_path), 'run this notebook from software/ (cannot find {})'.format(ckpt_path)

if torch.cuda.is_available():
    device = torch.device('cuda')
    cudnn.benchmark = True       # fixed 32x32 input -> let cuDNN pick the fastest conv kernels
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)

device: cpu


## Data (preloaded onto the device)

In [2]:
# Same normalization as HW3
mean = torch.tensor([0.491, 0.482, 0.447]).view(1, 3, 1, 1)
std  = torch.tensor([0.247, 0.243, 0.262]).view(1, 3, 1, 1)

def to_tensor(dataset):
    # Same result as transforms.ToTensor() + Normalize, but done once for the whole set:
    # uint8 NHWC -> float NCHW in [0, 1] -> normalized
    data, targets = dataset.data, np.array(dataset.targets)
    x = torch.from_numpy(data).permute(0, 3, 1, 2).contiguous().float().div(255)
    x = (x - mean) / std
    return x.to(device), torch.from_numpy(targets).long().to(device)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)

x_test, y_test = to_tensor(test_dataset)

print('test :', tuple(x_test.shape))


@torch.inference_mode()
def evaluate(model, x, y, batch_size=1000):
    # Top-1 accuracy (%) on preloaded tensors
    model.eval()
    correct = 0
    for i in range(0, len(x), batch_size):
        pred = model(x[i:i + batch_size]).argmax(dim=1)
        correct += (pred == y[i:i + batch_size]).sum().item()
    return 100.0 * correct / len(x)

test : (10000, 3, 32, 32)


c:\Users\yangg\miniconda3\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


## Load the trained FP32 ResNet-20 from HW3

In [3]:
model = resnet20_cifar().to(device)

checkpoint = torch.load(ckpt_path, map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

fp32_state = copy.deepcopy(model.state_dict())   # untouched FP32 weights, restored before every experiment
conv_names = [name for name, m in model.named_modules() if isinstance(m, nn.Conv2d)]   # same order as model.modules()

fp32_acc = evaluate(model, x_test, y_test)
print('checkpoint: epoch {}, best_prec {:.3f}%'.format(checkpoint['epoch'], checkpoint['best_prec']))
print('FP32 test accuracy: {:.2f}%'.format(fp32_acc))
print('{} conv layers: {}'.format(len(conv_names), conv_names))

checkpoint: epoch 199, best_prec 92.430%
FP32 test accuracy: 92.43%
21 conv layers: ['conv1', 'layer1.0.conv1', 'layer1.0.conv2', 'layer1.1.conv1', 'layer1.1.conv2', 'layer1.2.conv1', 'layer1.2.conv2', 'layer2.0.conv1', 'layer2.0.conv2', 'layer2.0.downsample.0', 'layer2.1.conv1', 'layer2.1.conv2', 'layer2.2.conv1', 'layer2.2.conv2', 'layer3.0.conv1', 'layer3.0.conv2', 'layer3.0.downsample.0', 'layer3.1.conv1', 'layer3.1.conv2', 'layer3.2.conv1', 'layer3.2.conv2']


## Weight quantizer from [Code8]

In [4]:
def weight_quantization(b):

    def uniform_quant(x, b):
        xdiv = x.mul((2 ** b - 1))
        xhard = xdiv.round().div(2 ** b - 1)
        return xhard

    class uq(torch.autograd.Function):

        @staticmethod
        def forward(ctx, input, alpha):
            input_d = input/alpha                          # weights are first divided by alpha
            input_c = input_d.clamp(min=-1, max=1)       # then clipped to [-1,1]
            sign = input_c.sign()
            input_abs = input_c.abs()
            input_q = uniform_quant(input_abs, b).mul(sign)
            ctx.save_for_backward(input, input_q)
            input_q_out = input_q.mul(alpha)               # rescale to the original range
            return input_q_out

    return uq.apply


class weight_quantize_fn(nn.Module):
    def __init__(self, w_bit):
        super(weight_quantize_fn, self).__init__()
        self.w_bit = w_bit-1
        self.weight_q = weight_quantization(b=self.w_bit)
        self.wgt_alpha = 0.0

    def forward(self, weight):
        weight_q = self.weight_q(weight, self.wgt_alpha)

        return weight_q


def quantize(w, alpha, w_bit):
    # alpha: a scalar, or a tensor that broadcasts against w (used to try many alphas at once)
    weight_quant = weight_quantize_fn(w_bit=w_bit)
    weight_quant.wgt_alpha = torch.as_tensor(alpha, dtype=w.dtype, device=w.device)
    return weight_quant(w)

## Choosing α

In [5]:
def alpha_mse(w, w_bit, n_grid=200):
    # Try all candidates at once: alphas [n_grid, 1] broadcast against the flattened weights [1, N]
    w_flat = w.reshape(1, -1)
    cands = torch.linspace(0.005, 1.0, n_grid, device=w.device) * w_flat.abs().max()
    err = (quantize(w_flat, cands[:, None], w_bit) - w_flat).pow(2).mean(dim=1)
    return cands[err.argmin()].item()


def quantize_model(model, w_bit):
    # Restore the FP32 weights, then quantize the weight of every Conv2d layer in place with its MSE-optimal alpha.
    # Returns the alpha used for each conv layer (model.modules() order, same as conv_names).
    model.load_state_dict(fp32_state)
    alphas = []
    with torch.no_grad():
        for layer in model.modules():
            if isinstance(layer, torch.nn.Conv2d):
                alpha = alpha_mse(layer.weight, w_bit)
                layer.weight.copy_(quantize(layer.weight, alpha, w_bit))
                alphas.append(alpha)
    return alphas


# Sanity check: every conv weight is now an integer multiple of alpha/(2^(b-1)-1) within the allowed range
for w_bit in (8, 4):
    alphas = quantize_model(model, w_bit)
    n_lv = 2 ** (w_bit - 1) - 1
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    w_int = [(m.weight / (a / n_lv)).round().long() for m, a in zip(convs, alphas)]
    n_distinct = [torch.unique(w).numel() for w in w_int]
    assert max(w.abs().max().item() for w in w_int) <= n_lv
    print('{}-bit: integer range [-{}, {}], distinct values per layer: {}'.format(w_bit, n_lv, n_lv, n_distinct))

print('\nlayer1[0].conv1, 4-bit integer weights of filter 0, input channel 0:')
print(w_int[1][0, 0])

8-bit: integer range [-127, 127], distinct values per layer: [116, 134, 146, 122, 173, 193, 159, 193, 182, 106, 159, 199, 176, 182, 218, 222, 193, 232, 185, 200, 200]
4-bit: integer range [-7, 7], distinct values per layer: [15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15]

layer1[0].conv1, 4-bit integer weights of filter 0, input channel 0:
tensor([[ 0,  0, -2],
        [ 1,  4,  2],
        [-3, -2, -3]])


## Quantize, evaluate, report

In [6]:
results = {'FP32': fp32_acc}
alphas_used = {}

t0 = time.time()
for w_bit in (8, 4):
    alphas_used[w_bit] = quantize_model(model, w_bit)
    results['{}-bit'.format(w_bit)] = evaluate(model, x_test, y_test)
print('done in {:.1f}s'.format(time.time() - t0))

print('\n{:<32s}{:>10s}{:>10s}'.format('model', 'test acc', 'drop'))
for k, acc in results.items():
    print('{:<32s}{:>9.2f}%{:>9.2f}%'.format(k, acc, fp32_acc - acc))

# Per-layer alpha, and the clipping ratio alpha / max|W| it corresponds to
w_max = np.array([fp32_state[n + '.weight'].abs().max().item() for n in conv_names])
print('\n{:<24s}{:>8s}{:>10s}{:>7s}{:>10s}{:>7s}'.format('layer', 'max|W|', '8b alpha', 'ratio', '4b alpha', 'ratio'))
for n, m, a8, a4 in zip(conv_names, w_max, alphas_used[8], alphas_used[4]):
    print('{:<24s}{:>8.4f}{:>10.4f}{:>7.3f}{:>10.4f}{:>7.3f}'.format(n, m, a8, a8 / m, a4, a4 / m))

done in 13.3s

model                             test acc      drop
FP32                                92.43%     0.00%
8-bit                               92.38%     0.05%
4-bit                               88.17%     4.26%

layer                     max|W|  8b alpha  ratio  4b alpha  ratio
conv1                     1.9492    1.9395  0.995    1.2865  0.660
layer1.0.conv1            1.0035    1.0035  1.000    0.5017  0.500
layer1.0.conv2            0.9260    0.9167  0.990    0.4815  0.520
layer1.1.conv1            1.0448    1.0448  1.000    0.4858  0.465
layer1.1.conv2            0.6157    0.6034  0.980    0.4095  0.665
layer1.2.conv1            0.6364    0.6364  1.000    0.4614  0.725
layer1.2.conv2            0.6154    0.6031  0.980    0.3723  0.605
layer2.0.conv1            0.6233    0.6109  0.980    0.3771  0.605
layer2.0.conv2            0.6384    0.6097  0.955    0.3288  0.515
layer2.0.downsample.0     1.3711    1.3573  0.990    0.7746  0.565
layer2.1.conv1            0.7240   

## Explanation and Observations

Using the regular floating point value weights, we achieve a baseline accuracy of 92.43%. However, when quantizing these weights into smaller-bit integers, we see some drops in accuracy. We iterated through different alpha values and selected the best one that kept accuracy high. For 8-bit integer quantization, the accuracy was able to maintain near the 32-bit floating point baseline, seeing a 0.05% drop to 92.38%. However, for the more aggressive 4-bit quantization, there was a larger and more noticeable drop in accuracy, dropping 4.26% to below 90%, at 88.17%. This may be because a 4-bit weight will more likely destroy the more detailed information of the previous 32-bit FP weights much more than the 8-bit quantization, which likely had enough bits left to keep most of the 32-bit weight information.